<a href="https://colab.research.google.com/github/n5n0/eskwelabs-portfolio/blob/main/py5_inspect_practice_bank_Crinan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Q16 · AVERAGE · SQL twin: d2 Q7

Select the customers with a KYC risk score of at least 80, showing `cust_id`, `age`, `occupation`, `kyc_risk_score`, highest score first. How many are there?

💡 *Hint:* `customers.loc[condition, [columns]]`, then `.sort_values(...)`.

### Q1 · EASY · SQL twin: Day 1 Activity 2, Query 1

How many rows and columns does `customers` have? In words, what is one row of `customers`?

💡 *Hint:* `.shape` gives (rows, columns). Then look at `.head()`.

# py5 · Inspect: Python Practice Bank (Day 5)

**DABC Sprint 2 · AML case · the same five tables you queried in SQL on Days 1–4**

### How to use this bank

- Questions are tiered **EASY · AVERAGE · DIFFICULT**. Work in order.
- **In class:** Practice block 1 (13:45) → **Q1–8**. Practice block 2 (14:40) → **Q9–20**, after Notebook 1 Your Turns 1–2. Your mentor will tell you the floor (what everyone finishes) for your room.
- **SQL twin** names the Day 1–4 question that asks the same thing. When both use the same table, your Python answer should give the same number as your SQL answer. If it doesn't, find out why: that's the lesson.
- Try each question yourself first. Then use AI to explain or check, following the sprint's AI protocol: paste the column names, never rows of data, and note it in your AI use log.
- ✍️ cells ask for a sentence, not code. Write it for the Head of AML.
- Your mentor has the answer key. Remaining questions are homework.

**Data you need** (Sprint 2 student folder, the same tables you queried in BigQuery on Days 1–4): `customers.csv`, `accounts.csv`, `transactions.csv`, `alerts.csv`, `investigations.csv` and `aml_analytical_extract.csv` (the Day 4 extract). Upload them to this Colab session, or put them in a Drive folder and add that folder to `DATA_DIRS` below.

In [1]:
import os
# List all files in the current working directory (/content)
print("Files in current directory:", os.listdir('.'))

Files in current directory: ['.config', 'accounts.csv', 'alerts.csv', 'customers.csv', 'investigations.csv', 'transactions.csv', 'sample_data']


In [2]:
import os

def find_csv_folders(start_path="/content/drive/MyDrive"):
    matching_folders = set()
    for root, dirs, files in os.walk(start_path):
        if "customers.csv" in files:
            matching_folders.add(root)
    return list(matching_folders)

print("Searching Google Drive for the dataset...")
paths = find_csv_folders()
if paths:
    print("Found 'customers.csv' in:")
    for p in paths:
        print(f"- {p}")
else:
    print("Could not find 'customers.csv' anywhere in your Google Drive.")
    print("Please make sure the folder is named correctly or uploaded inside your Drive.")

Searching Google Drive for the dataset...
Could not find 'customers.csv' anywhere in your Google Drive.
Please make sure the folder is named correctly or uploaded inside your Drive.


Once your drive is successfully mounted, you can locate the path to your folder containing the CSV files.
Right-click the folder in the file tree on the left, choose **Copy path**, and replace the placeholder in `DATA_DIRS` below (e.g., `"/content/drive/MyDrive/YOUR_FOLDER_NAME"`).

## Step 0 · Setup

Run these four cells in order with **Shift + Enter**. After a runtime restart, run them again.

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
print("pandas", pd.__version__, "· ready")

pandas 2.2.3 · ready


In [4]:
# ---------------------------------------------------------------------------
# CONFIGURATION — the only cell you should need to edit
# ---------------------------------------------------------------------------
USE_BIGQUERY = False                  # True = read the tables straight from BigQuery
PROJECT_ID   = "py5_inspect_practice_bank_Crinan"      # ← paste the project ID from your instructor
DATASET      = "aml_sprint2"

# Folders searched, in order, for the CSV files. Add your own Drive folder if needed.
DATA_DIRS = [".", "data", "/content", "/content/drive/MyDrive/csv datasets", "/content/drive/MyDrive/DABC24/aml_sprint2",]
print("USE_BIGQUERY =", USE_BIGQUERY)

USE_BIGQUERY = False


In [5]:
def load(name):

    if USE_BIGQUERY:
        from google.colab import auth
        from google.cloud import bigquery
        auth.authenticate_user()
        client = bigquery.Client(project=PROJECT_ID)
        frame = client.query(
            f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{name}`"
        ).to_dataframe()
        print(f"{name:<24} {len(frame):>7,} rows  (BigQuery)")
        return frame

    for folder in DATA_DIRS:
        path = os.path.join(folder, f"{name}.csv")
        if os.path.exists(path):
            frame = pd.read_csv(path)
            print(f"{name:<24} {len(frame):>7,} rows  ({path})")
            return frame

    raise FileNotFoundError(
        f"{name}.csv not found in {DATA_DIRS}. Upload it to this session "
        "or add its folder to DATA_DIRS in the configuration cell."
    )

In [7]:
customers      = load("customers")
accounts       = load("accounts")
transactions   = load("transactions")
alerts         = load("alerts")
investigations = load("investigations")

try:
    df         = load("aml_analytical_extract")
except FileNotFoundError as e:
    print("\n⚠️ Warning: aml_analytical_extract.csv was not found.")
    print("You can still answer Q1 to Q12 using the other loaded tables, but you will need to upload 'aml_analytical_extract.csv' to answer Q13 onwards.")

customers                 10,000 rows  (./customers.csv)
accounts                  12,028 rows  (./accounts.csv)
transactions              46,838 rows  (./transactions.csv)
alerts                     2,618 rows  (./alerts.csv)
investigations             1,623 rows  (./investigations.csv)
aml_analytical_extract    46,793 rows  (./aml_analytical_extract.csv)


## Block 1 · Python basics on the AML tables (Q1–8)

### Q1 · EASY · SQL twin: Day 1 Activity 2, Query 1

How many rows and columns does `customers` have? In words, what is one row of `customers`?

💡 *Hint:* `.shape` gives (rows, columns). Then look at `.head()`.

✍️ **One row of `customers` is one ______.**

*Your answer:*

In [ ]:
print(customers.shape)
customers.head()

(10000, 6)


,cust_id,age,occupation,declared_income,kyc_risk_score,customer_since
0,CUST-2fc59321,42,Business Owner,"3,364,115.84",67,2022-10-30
1,CUST-010b5ec1,29,Unemployed,NaN,54,2015-02-05
2,CUST-7d184195,60,Business Owner,"1,430,536.55",6,2023-05-20
3,CUST-87b646de,46,Business Owner,"1,716,363.97",28,2016-11-21
4,CUST-3a28ae55,67,Business Owner,"1,720,267.53",25,2019-05-07


### Q2 · EASY

Show the first 10 rows of `accounts`, then the last 5. Which column links an account to the customer who owns it?

💡 *Hint:* `.head(10)` and `.tail()`. Use `display()` to show two tables from one cell.

In [ ]:
# Show the first 10 rows and the last 5 rows of accounts
display(accounts.head(10))
display(accounts.tail(5))

,acct_id,cust_id,acct_type,open_date,initial_balance,acct_status
0,ACC-84237821,CUST-2fc59321,Checking,2024-05-08,"8,515.95",Closed
1,ACC-731d07c6,CUST-010b5ec1,Payroll,2016-11-19,"3,502.94",Active
2,ACC-373a10e4,CUST-7d184195,Business,2025-11-30,"27,606.87",Active
3,ACC-9d590a10,CUST-87b646de,Business,2020-04-25,"69,816.69",Active
4,ACC-31b1323d,CUST-3a28ae55,Business,2020-01-02,"11,091.49",Dormant
5,ACC-61223f80,CUST-ba313104,Checking,2025-11-30,"13,552.57",Active
6,ACC-36a123f5,CUST-909fdfb3,Savings,2016-12-26,"473,995.39",Active
7,ACC-bc25ba60,CUST-72bb8eb8,Checking,2025-11-30,"38,476.50",Dormant
8,ACC-e5b72916,CUST-a234d8d4,Savings,2021-10-23,"12,848.07",Active
9,ACC-ccf8a0da,CUST-ebed1c12,Business,2020-08-26,"152,846.33",Active


,acct_id,cust_id,acct_type,open_date,initial_balance,acct_status
12023,ACC-b495a6b9,CUST-85ec2f35,Business,2022-11-26,"23,071.05",Active
12024,ACC-95aa7891,CUST-06f47470,Savings,2019-07-11,"41,245.75",Active
12025,ACC-9a17dc2d,CUST-a364c904,Savings,2019-06-21,"12,653.17",Active
12026,ACC-3238565d,CUST-86277cb9,Checking,2025-11-30,"4,562.24",Active
12027,ACC-db44b19b,CUST-ceaef578,Checking,2021-01-17,"4,336.87",Active


### Q3 · EASY

Put the column names of `transactions` into a Python list called `tx_cols`. How many columns are there, and what is the **third** one?

💡 *Hint:* `.columns.tolist()`. Python counts from 0.

In [ ]:
# Put the column names of transactions into a Python list called tx_cols
tx_cols = transactions.columns.tolist()

# Display the list, length, and the third column (index 2)
print("Columns:", tx_cols)
print("Number of columns:", len(tx_cols))
print("The third column is:", tx_cols[2])

Columns: ['tx_id', 'orig_acct', 'dest_acct', 'amount', 'tx_datetime', 'channel', 'tx_type']
Number of columns: 7
The third column is: dest_acct


### Q4 · EASY

The Day 4 handoff note lists control totals for the extract: **46,793 rows · PHP 3,250,542,708.32 · 2,618 alerts · 1,623 investigations · 268 SARs**.

Store them in a dictionary called `expected` (keys: `rows`, `total_amount`, `alerts`, `investigations`, `sars`). Then loop over it and print each one on its own line with a thousands separator.

💡 *Hint:* `for name, value in expected.items():` and `f"{value:,}"`.

In [8]:
# Define the expected values dictionary
expected = {
    'rows': 46793,
    'total_amount': 3250542708.32,
    'alerts': 2618,
    'investigations': 1623,
    'sars': 268
}

# Loop over the dictionary and print each key-value pair with thousands separator
for name, value in expected.items():
    print(f"{name}: {value:,}")

rows: 46,793
total_amount: 3,250,542,708.32
alerts: 2,618
investigations: 1,623
sars: 268


### Q5 · AVERAGE

Write a function `pct(part, whole)` that returns `part` as a percentage of `whole`, rounded to 1 decimal place. Use it to answer: what percentage of transactions in the extract `df` were alerted (`was_alerted == 1`)?

💡 *Hint:* `def pct(part, whole):` … `return round(...)`. The sum of a 0/1 column counts the 1s.

In [9]:
def pct(part, whole):
    return round((part / whole) * 100, 1)

# Count the alerted transactions
alerted_count = df['was_alerted'].sum()
total_count = len(df)

# Calculate and print the percentage
percentage_alerted = pct(alerted_count, total_count)
print(f"Alerted transactions: {alerted_count:,} out of {total_count:,} ({percentage_alerted}%)")

Alerted transactions: 2,381 out of 46,793 (5.1%)


### Q6 · AVERAGE

Put the five source tables in a dictionary: `tables = {"customers": customers, ...}`. Loop over it and print one line per table: its name, number of rows and number of columns.

💡 *Hint:* `rows, cols = table.shape` unpacks the pair.

In [ ]:
# Your code here


### Q7 · AVERAGE · SQL twin: Day 1 Activity 2, stretch S1

Is `acct_id` really unique in `accounts`? Compare the number of rows with the number of distinct `acct_id` values. How many duplicate rows are there?

💡 *Hint:* `len(accounts)` vs `accounts["acct_id"].nunique()`.

In [ ]:
# Your code here


### Q8 · DIFFICULT · SQL twin: Day 1 Activity 2, stretch S1 (all tables)

Do the uniqueness check for **all five** tables in one loop, using `tables` from Q6. Collect the results in a DataFrame with columns `table`, `key`, `total_rows`, `distinct_keys`, `duplicate_rows`.

Which two tables have duplicate keys?

💡 *Hint:* Make a second dictionary of key columns (`customers` → `cust_id`, …). Append one dict per table to a list, then `pd.DataFrame(list_of_dicts)`.

In [ ]:
# Your code here


## Block 2 · Inspecting a DataFrame is SQL you already know (Q9–20)

You can use one-condition filters (`df.loc[df["amount"] >= 500_000]`) and sorting (`.sort_values()`). Combined conditions with `&` and `|` come on Day 6.

### Q9 · EASY

Run `customers.info()`. Which column holds dates but is stored as text? Which numeric column has missing values, and how many?

💡 *Hint:* Read the Non-Null Count and Dtype columns. A text column shows as `object`.

In [ ]:
# Run customers.info() to answer Q9
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cust_id          10000 non-null  object 
 1   age              10000 non-null  int64  
 2   occupation       9620 non-null   object 
 3   declared_income  9790 non-null   float64
 4   kyc_risk_score   10000 non-null  int64  
 5   customer_since   10000 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 468.9+ KB


1.   customer_since - Holds date strings
2.   declared_income - It is a numeric column (float64) and has 210 missing values



### Q10 · EASY · SQL twin: d3a Q5

How many distinct `channel` values appear in `transactions`? List them with `.value_counts(dropna=False)`. How many *real* channels do you think there are?

💡 *Hint:* `.nunique()` ignores missing values; `dropna=False` shows them.

In [ ]:
# Show distinct channel values with counts, including missing values
display(transactions['channel'].value_counts(dropna=False))

,count
channel,
Online,7548
Branch,7530
PESONet,7139
Mobile,6443
InstaPay,6443
ATM,6417
pesonet,444
Pesonet,437
mobile,415


### Q11 · EASY · SQL twin: Day 1 Activity 2, Query 4

Which alert rule produces the most alerts, and what share of all alerts is that? Use `.value_counts(normalize=True)`.

In [ ]:
# Show the percentage share of each alert rule
display(alerts['rule_name'].value_counts(normalize=True))

,proportion
rule_name,
Large_Amount_CTR,0.59
High_Velocity_Count,0.18
Near_Threshold_Structuring,0.14
Dormant_Reactivation,0.05
Rapid_Movement,0.05


In [ ]:
# 1. Preview the first 5 rows of the customers table
print("=== Customers Preview ===")
display(customers.head())

# 2. Check the column types and non-null counts of the alerts table
print("\n=== Alerts Schema & Info ===")
alerts.info()

# 3. View key statistics for numeric columns in the transactions table
print("\n=== Transactions Summary ===")
display(transactions.describe())

=== Customers Preview ===


,cust_id,age,occupation,declared_income,kyc_risk_score,customer_since
0,CUST-2fc59321,42,Business Owner,"3,364,115.84",67,2022-10-30
1,CUST-010b5ec1,29,Unemployed,NaN,54,2015-02-05
2,CUST-7d184195,60,Business Owner,"1,430,536.55",6,2023-05-20
3,CUST-87b646de,46,Business Owner,"1,716,363.97",28,2016-11-21
4,CUST-3a28ae55,67,Business Owner,"1,720,267.53",25,2019-05-07



=== Alerts Schema & Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2618 entries, 0 to 2617
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   alert_id        2618 non-null   object
 1   tx_id           2618 non-null   object
 2   rule_name       2618 non-null   object
 3   alert_datetime  2618 non-null   object
 4   alert_score     2618 non-null   object
dtypes: object(5)
memory usage: 102.4+ KB

=== Transactions Summary ===


,amount
count,"46,838.00"
mean,"69,439.48"
std,"255,197.59"
min,"-166,901.90"
25%,611.43
50%,"1,353.14"
75%,"3,616.07"
max,"7,128,015.98"


### Q12 · EASY · SQL twin: Day 1 Activity 2, Query 5

What are the possible values of `investigation_result`, and how many cases fall into each? Include missing values.

In [ ]:
# Your code here


### Q13 · AVERAGE

Count the missing values in every column of `customers`, and show them as a percentage of rows too, in one small table. Which column is worst?

💡 *Hint:* `.isna().sum()` and `.isna().mean()`. Build the table with `pd.DataFrame({"missing": ..., "missing_pct": ...})`.

In [ ]:
# Calculate missing counts and percentages for each column in customers
missing_count = customers.isna().sum()
missing_pct = customers.isna().mean() * 100

# Combine them into a single DataFrame
missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
})

display(missing_summary)

,missing_count,missing_pct
cust_id,0,0.00
age,0,0.00
occupation,380,3.80
declared_income,210,2.10
kyc_risk_score,0,0.00
customer_since,0,0.00


Which column is the Worst?  Occupation is the worst with 380 missing_count.

### Q14 · AVERAGE · SQL twin: d2 Q3

Show the 10 largest transactions in `transactions` (`tx_id`, `orig_acct`, `dest_acct`, `amount`, `channel`), largest first. How many of the 10 go to an account **outside** the bank (`EXT-`)?

💡 *Hint:* `.sort_values("amount", ascending=False).head(10)`. The first three characters of a text column: `.str[:3]`.

In [ ]:
# Show the 10 largest transactions, displaying the requested columns
largest_tx = transactions[['tx_id', 'orig_acct', 'dest_acct', 'amount', 'channel']].sort_values(by='amount', ascending=False).head(10)
display(largest_tx)

,tx_id,orig_acct,dest_acct,amount,channel
20900,TX-8ca744ad6b,ACC-c0dd8936,EXT-228c1d52,"7,128,015.98",branch
762,TX-eaf36dfd0d,ACC-8241a18f,EXT-f4853c11,"6,777,213.11",online
7444,TX-f29bd6727c,ACC-99091797,ACC-e1b38449,"6,724,057.08",PESONet
28362,TX-4192c30939,ACC-6d1e33d6,EXT-b31b9958,"6,206,735.99",Branch
10863,TX-a9643b7b02,ACC-da24faa7,EXT-73ae1c92,"4,662,387.14",Branch
34761,TX-17cc08b377,ACC-b343b3f6,EXT-9389e5a1,"4,404,915.68",Online
43518,TX-2dcab503b5,ACC-ac69da47,EXT-da450379,"4,076,245.59",Online
2583,TX-ce6ce10991,ACC-75880ad4,ACC-5e93d020,"3,953,285.84",PESONet
43853,TX-8f49766a72,ACC-ba73f0c4,EXT-75808971,"3,345,729.75",Online
37060,TX-18df943a35,ACC-ac69da47,EXT-67e402aa,"3,284,328.14",Online


### Q15 · AVERAGE · SQL twin: d2 Q6

How many transactions are worth at least PHP 500,000? Count them in the raw `transactions` table, then in the extract `df`. Why do the two numbers differ?

💡 *Hint:* A comparison gives True/False for every row; `.sum()` counts the Trues.

In [ ]:
# Load datasets if they are not already in the environment
try:
    transactions
except NameError:
    import pandas as pd
    transactions = pd.read_csv('transactions.csv')

try:
    df
except NameError:
    import pandas as pd
    df = pd.read_csv('aml_analytical_extract.csv')

# Count transactions >= 500,000 in raw transactions table
raw_count = (transactions['amount'] >= 500000).sum()
print(f"Transactions >= 500,000 in raw transactions: {raw_count:,}")

# Count transactions >= 500,000 in the analytical extract df
extract_count = (df['amount'] >= 500000).sum()
print(f"Transactions >= 500,000 in analytical extract df: {extract_count:,}")

print(f"Difference: {raw_count - extract_count:,}")

Transactions >= 500,000 in raw transactions: 1,542
Transactions >= 500,000 in analytical extract df: 1,540
Difference: 2


✍️ **Why do the two counts differ?**

Transactions >= 500,000 in raw transactions: 1,542
Transactions >= 500,000 in analytical extract df: 1,540
Difference: 2

### Q16 · AVERAGE · SQL twin: d2 Q7

Select the customers with a KYC risk score of at least 80, showing `cust_id`, `age`, `occupation`, `kyc_risk_score`, highest score first. How many are there?

💡 *Hint:* `customers.loc[condition, [columns]]`, then `.sort_values(...)`.

In [ ]:
# Load customers dataset if it is not already in the environment
try:
    customers
except NameError:
    import pandas as pd
    customers = pd.read_csv('/content/customers.csv')

# Select customers with a KYC risk score of at least 80
high_risk_customers = customers.loc[customers['kyc_risk_score'] >= 80, ['cust_id', 'age', 'occupation', 'kyc_risk_score']].sort_values(by='kyc_risk_score', ascending=False)

# Display the results
display(high_risk_customers)

# Print the total count of these customers
print(f"\nNumber of customers with KYC risk score >= 80: {len(high_risk_customers)}")

,cust_id,age,occupation,kyc_risk_score
1058,CUST-c70753bf,42,Unemployed,93
4115,CUST-45b09eb4,24,Student,93
2913,CUST-e663c347,35,Unemployed,91
8675,CUST-8ee26a7d,55,Unemployed,91
9426,CUST-2a44cc43,35,Unemployed,90
1136,CUST-21af1056,22,Student,89
1259,CUST-16e969bd,37,Unemployed,88
2688,CUST-b62881a9,49,BUSINESS OWNER,88
9074,CUST-1ad786e4,59,Business Owner,87
9534,CUST-b64ac42e,69,Business Owner,86



Number of customers with KYC risk score >= 80: 32


### Q17 · AVERAGE · SQL twin: d2 Q9

How many customers have a **missing** `declared_income`? How many have a declared income of exactly **0**?

In [ ]:
# Your code here


✍️ **Why should these two groups be treated differently on Day 6?**

*Your answer:*

### Q18 · AVERAGE

Run `.describe()` on `amount` in the extract. How many times bigger is the mean than the median?

In [ ]:
# Your code here


✍️ **Which number would you quote to the Head of AML as a typical transaction, and why?**

*Your answer:*

### Q19 · DIFFICULT

**Grain check.** In the extract, how many **rows** have a missing `occupation`? How many **distinct customers** (`orig_cust_id`) do those rows belong to?

💡 *Hint:* Save the condition in a variable first: `no_occ = df["occupation"].isna()`. Then `.sum()` it, and use it inside `.loc` to count distinct customers.

In [ ]:
# Your code here


✍️ **Write one sentence about this for the Head of AML that she could not misread.**

*Your answer:*

### Q20 · DIFFICULT

Write a function `quick_profile(frame)` that returns a DataFrame with one row per column of `frame` and these columns: `column`, `dtype`, `missing`, `missing_pct`, `unique`. Run it on `alerts`.

Which column looks like a number but is stored as text, and why?

💡 *Hint:* Loop over `frame.columns`, build a dict per column, append it to a list, return `pd.DataFrame(rows)`. Notebook 1 Step 3 has the same pattern for one column.

In [ ]:
# Your code here


---
### Done?

Unfinished questions are homework. Bring the one you were most stuck on to tomorrow's opener, along with what you tried.